In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from Winnipeg to Lisbon on March 31st, if no direct flights are available, get me the next best option")]},
    config
    )

In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from Winnipeg to Lisbon on March 31st, if no direct flights are available, get me the next best option', additional_kwargs={}, response_metadata={}, id='2ad8583d-b0b7-4559-8d6e-faa8a365b1b3'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 792, 'prompt_tokens': 1079, 'total_tokens': 1871, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DzUTwo8vrreXSeMu0sk9KDPjqgD05', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f43b7-ddb3-7321-9a92-11b16a847bf7-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'Winnipeg', 'flyTo': 'Lisbon',

In [12]:
print(response["messages"][-1].content)

I found there are no direct Winnipeg (YWG) to Lisbon (LIS) flights on March 31. Here are the best next options:

| Itinerary (Outlet) | Route | Outbound times (local) | Stops | Cabin | Price | Booking |
|---|---|---|---|---|---|---|
| Cheapest (457 EUR) | YWG → YYC → YYZ → PDL → LIS | 31 Mar 07:50 → 01 Apr 11:40 | 3 | Economy | 457 EUR | https://kiwi.com/u/58y4kq |
| Shortest (465 EUR) | YWG → YYZ → PDL → LIS | 31 Mar 12:20 → 01 Apr 11:40 | 2 | Economy | 465 EUR | https://kiwi.com/u/gpx29u |
| Notable option (471 EUR) | YWG → YYC → YYZ → LIS | 31 Mar 06:15 → 01 Apr 05:55 | 2 | Economy | 471 EUR | https://kiwi.com/u/qdbdtt8 |

Additional nearby options (higher duration or price) include several with 1–3 stops, around 472–500 EUR, for example via YYZ and LIS with various routings. If you’d like, I can book one of these for you or keep searching for any direct-windows that may pop up.

Fun tip for Lisbon: You’ll love a pastéis de nata (custard tart) fresh from the oven. They’re a beloved 